# Lab 2 — MCP on the wire (EP13)

In Lab 1 we talked to the server through `ClientSession` — which hides every message.
In this lab we are the client: we start the server and **type JSON-RPC 2.0 messages one line at a time** into its stdin, and read the raw replies from stdout (§21.4)

**What you'll do**
1. Start the server as a child process — the stdio transport is one JSON message per line
2. **Handshake:** `initialize` → agree on a version and capabilities → `notifications/initialized`
3. **Request / response:** `tools/list`, `tools/call` — matched by `id`
4. **Two levels of error:** a tool error (`isError`) and a protocol error (`-32602`)
5. **Notifications:** progress (`notifications/progress`) and a changed tool list (`list_changed`)
6. **Shutdown:** close the pipe, and the server shuts itself down

This lab calls no model, so it needs no API key and costs nothing.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §21.2.3, §21.4, §21.5.2 — CC BY-SA 4.0

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| Packages | `mcp` (for the server only — we write the client side ourselves) |

In [ ]:
%pip install -q "mcp==1.30.0"

## 1. The same server + two new tools

`search_papers` and the resource `paper://{paper_id}` from Lab 1 · plus two new tools, to see two more kinds of message
- `index_library` — a slow job that **reports progress** (§21.4.4)
- `enable_citations` — adds a `cite_paper` tool during the session and **announces that the tool list changed** (§21.5.2)

In [ ]:
%%writefile paper_server.py
import json, asyncio
from mcp.server.fastmcp import FastMCP, Context

mcp = FastMCP("cogito-papers")

PAPERS = {  # teaching data: real arXiv ids, titles and years; short summaries written for this lab
    "2211.14730": {"title": "A Time Series is Worth 64 Words: Long-term Forecasting with Transformers",
                   "authors": "Nie et al.", "year": 2023, "summary": "PatchTST: split each series into patches, one channel at a time"},
    "2310.06625": {"title": "iTransformer: Inverted Transformers Are Effective for Time Series Forecasting",
                   "authors": "Liu et al.", "year": 2024, "summary": "attention across variables instead of across time steps"},
    "2310.10688": {"title": "A decoder-only foundation model for time-series forecasting",
                   "authors": "Das et al.", "year": 2024, "summary": "TimesFM: one pretrained model that forecasts many series zero-shot"},
    "2012.07436": {"title": "Informer: Beyond Efficient Transformer for Long Sequence Time-Series Forecasting",
                   "authors": "Zhou et al.", "year": 2021, "summary": "sparse attention to make long forecasts cheaper"},
    "2205.13504": {"title": "Are Transformers Effective for Time Series Forecasting?",
                   "authors": "Zeng et al.", "year": 2023, "summary": "a simple linear model beats several transformers"},
}

@mcp.tool()
def search_papers(query: str, max_results: int = 3) -> str:
    """Search the paper library by keyword in the title or summary.
    Use this to find papers on a topic. Returns a JSON list of {id, title, year}, best match first."""
    words = query.lower().split()
    scored = []
    for pid, p in PAPERS.items():
        score = sum(w in (p["title"] + " " + p["summary"]).lower() for w in words)
        if score:
            scored.append((score, {"id": pid, "title": p["title"], "year": p["year"]}))
    scored.sort(key=lambda s: -s[0])
    return json.dumps([paper for _, paper in scored[:max_results]])

@mcp.resource("paper://{paper_id}")
def get_paper(paper_id: str) -> str:
    """One paper's full record, as JSON."""
    return json.dumps({"id": paper_id, **PAPERS[paper_id]})

In [ ]:
%%writefile -a paper_server.py

@mcp.tool()
async def index_library(ctx: Context) -> str:
    """Rebuild the search index. Slow; reports progress."""
    n = len(PAPERS)
    for i in range(1, n + 1):
        await asyncio.sleep(0.3)
        await ctx.report_progress(i, n, f"indexed {i}/{n} papers")
    return f"indexed {n} papers"

In [ ]:
%%writefile -a paper_server.py

@mcp.tool()
async def enable_citations(ctx: Context) -> str:
    """Unlock the cite_paper tool for this session."""
    def cite_paper(paper_id: str) -> str:
        """Format a short citation for one paper, by its arXiv id."""
        p = PAPERS[paper_id]
        return f'{p["authors"]} ({p["year"]}). {p["title"]}. arXiv:{paper_id}'
    mcp.add_tool(cite_paper)
    await ctx.session.send_tool_list_changed()
    return "cite_paper is now available"

if __name__ == "__main__":
    mcp.run()

## 2. Start the server and wire up the pipes ourselves

The stdio transport (§21.2.2): the client starts the server as a child process and talks over **stdin / stdout**
One message = one line of JSON · the server's log goes to stderr (we keep it in a file)

In [ ]:
import sys, json, subprocess

server = subprocess.Popen([sys.executable, "paper_server.py"],
                          stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                          stderr=open("server.log", "w"), text=True)

In [ ]:
def send(msg):
    server.stdin.write(json.dumps(msg) + "\n")
    server.stdin.flush()
    print("➡️ ", json.dumps(msg)[:300])

def recv():
    msg = json.loads(server.stdout.readline())
    print("⬅️ ", json.dumps(msg)[:300])
    return msg

## 3. The handshake: `initialize`

The first message must be `initialize` (§21.4.2): a protocol version · the client's capabilities · the client's name
We send version `2024-11-05`, as in the book's example

In [ ]:
send({"jsonrpc": "2.0", "id": 1, "method": "initialize",
      "params": {"protocolVersion": "2024-11-05", "capabilities": {},
                 "clientInfo": {"name": "by-hand", "version": "0.1"}}})
init = recv()
print("version:", init["result"]["protocolVersion"])
print("server capabilities:", init["result"]["capabilities"])

In [ ]:
send({"jsonrpc": "2.0", "method": "notifications/initialized"})

## 4. Requests and responses

Every request carries an `id` · its response comes back with the same `id` (§21.4.1) — like a restaurant ticket

In [ ]:
send({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})
tools = recv()["result"]["tools"]
for t in tools:
    print("🔧", t["name"], "· required:", t["inputSchema"].get("required", []))

In [ ]:
send({"jsonrpc": "2.0", "id": 42, "method": "tools/call",
      "params": {"name": "search_papers", "arguments": {"query": "linear", "max_results": 2}}})
reply = recv()
print("id:", reply["id"], "· isError:", reply["result"]["isError"])
print(reply["result"]["content"][0]["text"])

## 5. Two levels of error

| Level | When | What it looks like |
|---|---|---|
| **tool** | the tool fails (bad argument, missing data) | a normal `result` with `isError: true` — the text can go back to the model |
| **protocol** | the request itself is wrong | no `result` · an `error` with a code, e.g. `-32602` invalid params (§21.4.3) |

In [ ]:
send({"jsonrpc": "2.0", "id": 43, "method": "tools/call",
      "params": {"name": "search_papers", "arguments": {}}})
bad_tool = recv()
print("has result:", "result" in bad_tool, "· isError:", bad_tool["result"]["isError"])

In [ ]:
send({"jsonrpc": "2.0", "id": 44, "method": "resources/read", "params": {}})
bad_request = recv()
print("has result:", "result" in bad_request, "· error:", bad_request["error"])

## 6. Notifications: progress, and a changed tool list

A request with `_meta.progressToken` → the server sends `notifications/progress` before the reply (§21.4.4)
so we keep reading until the message with our `id` arrives

In [ ]:
def recv_until(request_id):
    while True:
        msg = recv()
        if msg.get("id") == request_id:
            return msg

send({"jsonrpc": "2.0", "id": 45, "method": "tools/call",
      "params": {"name": "index_library", "arguments": {}, "_meta": {"progressToken": "index-op-1"}}})
print(recv_until(45)["result"]["content"][0]["text"])

In [ ]:
send({"jsonrpc": "2.0", "id": 46, "method": "tools/call", "params": {"name": "enable_citations", "arguments": {}}})
recv_until(46)
send({"jsonrpc": "2.0", "id": 47, "method": "tools/list"})
print([t["name"] for t in recv_until(47)["result"]["tools"]])

## 7. Shutdown

Over stdio, the client ends the connection by **closing the server's stdin**, then waiting for it to exit

In [ ]:
server.stdin.close()
print("exit code:", server.wait(timeout=10))
print("server saw:", sorted({line.split("type ")[1].strip() for line in open("server.log") if "request of type" in line}))

## Summary

| Message | What we saw |
|---|---|
| `initialize` (id 1) | agreed on version 2024-11-05 · the server declares tools, resources, prompts |
| `notifications/initialized` | no id · no reply |
| `tools/call` (id 42) | reply id 42 · `content` + `isError: false` |
| a failing tool (id 43) | a normal `result` + `isError: true` — the model can read it |
| a bad request (id 44) | `error` code `-32602` — no `result` |
| `progressToken` (id 45) | `notifications/progress` 1/5 … 5/5, then the reply |
| `enable_citations` (id 46) | `notifications/tools/list_changed` → `tools/list` includes `cite_paper` |
| close stdin | the server exits with code 0 |

**Exercises**
1. Send `tools/call` before `initialize` — what does the server answer?
2. Send a `resources/read` for `paper://2310.06625` yourself and find `contents` in the reply
3. Send two requests (id 50, 51) back to back before reading, and see what order the replies come in — how does `id` help?

**Next — Lab 3:** we write a host for our agent that connects to two servers at once and stops to ask before calling a tool that can't be undone